# Smart LogiTrack - Système Prédictif de Transport Urbain (ETA)

### The Process

* `Ingest` Read the file (Parquet/CSV).

* `Clean` Handle Nulls, filter outliers.

* `Feature Eng` Create new variables.

* `Split` Separate Train / Test.

* `Train` model.fit()

* `Evaluate` Compare the prediction to reality.

In [ ]:
import os
import findspark
from pyspark.sql import SparkSession

# Remplace ce chemin par le vrai chemin où tu as installé Java 11
os.environ["JAVA_HOME"] = "C:/Program Files/Eclipse Adoptium/jdk-11.0.18.10-hotspot/"

# Ensuite tu lances Spark

findspark.init()
from pyspark.sql import SparkSession
# ...
# --- start up 
spark = SparkSession.builder \
    .appName("ParquetName") \
    .master("local[*]") \
    .getOrCreate()



In [1]:
import pandas as pd

# --- loading data
df = pd.read_parquet('data/bronze_taxi.parquet')
df.head(5)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


### info

In [2]:
print("=== INFO ===")
df.info()


=== INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_f

In [3]:
# --- possible val : obj col
df_obj=df.select_dtypes(['object'])
df_obj.head(100)
df['store_and_fwd_flag'].unique()

array(['N', 'Y', None], dtype=object)

## Traget variable treatement

In [ ]:
from datetime import datetime

df_copy=df.copy()
fmt="%Y-%m-%d %H:%M:%S"

# pick_up=datetime.strptime(df_copy["tpep_pickup_datetime"], fmt)
# drop_off=datetime.strptime(df_copy["tpep_dropoff_datetime"], fmt)
pick_up=df_copy["tpep_pickup_datetime"]
drop_off=df_copy["tpep_dropoff_datetime"]

trip_duration=drop_off-pick_up
print(f"trip duration : Date object --> {trip_duration.head(3)}")

# duration_sec=trip_duration.dt.total_seconds()
# print(f"trip duration : seconds --> {duration_sec.head(3)}")

duration_min=trip_duration.dt.total_seconds()
print(f"trip duration : seconds --> {duration_min.head(3)}")

trip duration : Date object --> 0   0 days 00:08:21
1   0 days 00:02:33
2   0 days 00:01:57
dtype: timedelta64[us]
trip duration : seconds --> 0    501.0
1    153.0
2    117.0
dtype: float64
trip duration : seconds --> 0    501.0
1    153.0
2    117.0
dtype: float64


## Correlation

## Importations et Initialisation Spark

In [ ]:
import os
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, unix_timestamp, hour, dayofweek, month, to_timestamp
from pyspark.sql.types import DoubleType, IntegerType

# Configuration de l'environnement graphique
%matplotlib inline
sns.set(style="whitegrid")

# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("SmartLogiTrack_EDA_Notebook") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## Chargement des Données (Bronze)

In [ ]:
# Chemin vers le fichier Parquet (Volume Docker ou local)
file_path = "data/bronze_taxi.parquet"

# Lecture du fichier Parquet
df_bronze = spark.read.parquet(file_path)

print(f"Nombre de lignes : {df_bronze.count()}")
print("Schéma des données :")
df_bronze.printSchema()

## Nettoyage et Feature Engineering (Vers Silver)

In [ ]:
# 1. Conversion des dates et calcul de la durée (Target)
df_silver = df_bronze.withColumn(
    "duration_minutes",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
)

# 2. Application des filtres métier (Brief)
# - Distance entre 0 et 200 miles
# - Durée positive
# - Nombre de passagers > 0
df_silver = df_silver.filter(
    (col("trip_distance") > 0) & 
    (col("trip_distance") <= 200) &
    (col("duration_minutes") > 0) & 
    (col("duration_minutes") < 300) & # Filtre outlier durée extrême (>5h)
    (col("passenger_count") > 0)
)

# 3. Création des features temporelles
df_silver = df_silver.withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
                     .withColumn("day_of_week", dayofweek("tpep_pickup_datetime")) \
                     .withColumn("month", month("tpep_pickup_datetime"))

# Aperçu
df_silver.select("tpep_pickup_datetime", "duration_minutes", "trip_distance", "pickup_hour").show(5)

## EDA - Analyse Exploratoire (Visualisation)

In [ ]:
# On prend un échantillon de 10% pour l'analyse graphique afin de ne pas surcharger la mémoire
pdf_sample = df_silver.sample(fraction=0.1, seed=42).toPandas()

# 1. Distribution de la variable cible (Durée)
plt.figure(figsize=(10, 5))
sns.histplot(pdf_sample['duration_minutes'], bins=50, kde=True, color='blue')
plt.title("Distribution de la Durée des Trajets (Minutes)")
plt.xlim(0, 60) # Focus sur les trajets de moins d'une heure
plt.show()

# 2. Relation Distance vs Durée
plt.figure(figsize=(10, 5))
sns.scatterplot(x='trip_distance', y='duration_minutes', data=pdf_sample, alpha=0.3)
plt.title("Corrélation : Distance vs Durée")
plt.xlabel("Distance (Miles)")
plt.ylabel("Durée (Minutes)")
plt.xlim(0, 30)
plt.ylim(0, 100)
plt.show()

# 3. Durée moyenne par heure de la journée
plt.figure(figsize=(12, 6))
sns.lineplot(x='pickup_hour', y='duration_minutes', data=pdf_sample, estimator='mean', errorbar=None, marker='o')
plt.title("Durée moyenne des trajets par heure (Détection Heures de Pointe)")
plt.xticks(range(0, 24))
plt.show()

## Analyse SQL Avancée (Prototypage pour API)

In [ ]:
# Création d'une vue temporaire pour utiliser Spark SQL
df_silver.createOrReplaceTempView("silver_taxi_trips")

# Requête 1 : Analyse des paiements (Brief: /analytics/payment-analysis)
query_payment = """
    SELECT 
        payment_type, 
        COUNT(*) as total_trips, 
        ROUND(AVG(duration_minutes), 2) as avg_duration 
    FROM silver_taxi_trips 
    GROUP BY payment_type 
    ORDER BY total_trips DESC
"""
spark.sql(query_payment).show()

# Requête 2 : Analyse horaire (Brief: /analytics/avg-duration-by-hour)
query_hourly = """
    SELECT 
        pickup_hour, 
        ROUND(AVG(duration_minutes), 2) as avg_duration 
    FROM silver_taxi_trips 
    GROUP BY pickup_hour 
    ORDER BY pickup_hour ASC
"""
spark.sql(query_hourly).show(5)

## Préparation au Machine Learning (Training)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Sélection des features
features = ['trip_distance', 'passenger_count', 'pickup_hour', 'day_of_week']
target = 'duration_minutes'

# Conversion Spark -> Pandas pour Scikit-Learn (Sur l'échantillon ou dataset complet si RAM suffisante)
# Ici on utilise l'échantillon précédent pour la rapidité du notebook
X = pdf_sample[features]
y = pdf_sample[target]

# Split Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

## Entraînement et Évaluation du Modèle

In [ ]:
# Initialisation du modèle
rf_model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)

# Entraînement
print("Début de l'entraînement...")
rf_model.fit(X_train, y_train)
print("Entraînement terminé.")

# Prédiction
y_pred = rf_model.predict(X_test)

# Métriques
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f} minutes")
print(f"R2 Score: {r2:.4f}")

# Feature Importance
import numpy as np
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\nImportance des features :")
for f in range(X.shape[1]):
    print(f"{features[indices[f]]}: {importances[indices[f]]:.4f}")

## Sérialisation (Test de sauvegarde)

In [ ]:
# Test de sauvegarde pour vérifier que ça fonctionnera dans le script final
model_path = "models/model_notebook_test.pkl"
os.makedirs("models", exist_ok=True)

joblib.dump(rf_model, model_path)
print(f"Modèle sauvegardé avec succès sous : {model_path}")

# Test de rechargement
loaded_model = joblib.load(model_path)
test_pred = loaded_model.predict(X_test.iloc[0:1])
print(f"Test prédiction (1 ligne) : {test_pred[0]:.2f} minutes")

## Arrêt de Spark

In [ ]:
spark.stop()

***

In [4]:
import pandas as pd
import io

# Configuration
FILE_NAME = "data/bronze_taxi.parquet"
OUTPUT_FILE = "context_pour_ai_studio.txt"

def generate_report():
    try:
        df = pd.read_parquet(FILE_NAME)
        
        # Buffer pour écrire le rapport
        buffer = io.StringIO()
        
        buffer.write(f"--- RAPPORT CONTEXTUEL DE DONNÉES : {FILE_NAME} ---\n")
        buffer.write("Ce rapport décrit la structure et la distribution des données pour un projet Data Engineering.\n\n")
        
        # 1. MÉTADONNÉES GÉNÉRALES
        buffer.write(f"## 1. VOLUMÉTRIE\n")
        buffer.write(f"- Nombre de lignes : {len(df)}\n")
        buffer.write(f"- Nombre de colonnes : {len(df.columns)}\n")
        buffer.write(f"- Usage Mémoire (approx) : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB\n\n")
        
        # 2. ANALYSE DÉTAILLÉE PAR COLONNE
        buffer.write("## 2. DÉTAILS DES VARIABLES ET DISTRIBUTIONS\n")
        
        for col in df.columns:
            dtype = df[col].dtype
            buffer.write(f"\n### Variable: '{col}' (Type: {dtype})\n")
            
            # Gestion des valeurs manquantes
            nulls = df[col].isnull().sum()
            if nulls > 0:
                buffer.write(f"- Valeurs manquantes : {nulls} ({nulls/len(df):.2%})\n")
            
            # Analyse Spécifique selon le type
            if pd.api.types.is_numeric_dtype(dtype):
                stats = df[col].describe()
                buffer.write(f"- Stats : Min={stats['min']}, Max={stats['max']}, Moyenne={stats['mean']:.2f}\n")
                
                # Détection de potentielles anomalies (valeurs négatives pour des montants)
                negatives = (df[col] < 0).sum()
                if negatives > 0 and "amount" in col:
                    buffer.write(f"- NOTE : Contient {negatives} valeurs négatives (potentiels retours/erreurs).\n")
                    
            elif pd.api.types.is_datetime64_any_dtype(dtype):
                buffer.write(f"- Période temporelle : De {df[col].min()} à {df[col].max()}\n")
                
            else: # Objets / Catégories / IDs
                unique_count = df[col].nunique()
                buffer.write(f"- Cardinalité : {unique_count} valeurs uniques\n")
                if unique_count < 50:
                    top_vals = df[col].value_counts().head(5).to_dict()
                    buffer.write(f"- Top valeurs fréquentes : {top_vals}\n")
        
        # 3. ÉCHANTILLON DE DONNÉES
        buffer.write("\n## 3. ÉCHANTILLON BRUT (JSON)\n")
        buffer.write("Voici 3 lignes représentatives pour comprendre le format exact :\n")
        buffer.write(df.head(3).to_json(orient='records', date_format='iso', lines=True))
        
        # Sauvegarde
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            f.write(buffer.getvalue())
            
        print(f"✅ Succès ! Le rapport a été généré dans '{OUTPUT_FILE}'.")

    except Exception as e:
        print(f"❌ Erreur : {e}")

if __name__ == "__main__":
    generate_report()

❌ Erreur : [Errno 2] No such file or directory: 'data/bronze_taxi.parquet'
